# 動画のシーン変化を一覧画像にする

動画の先頭と大きく画面が変わる場面を抽出し、コンタクトシートと個別フレームの ZIP を作ります。動画の構成を見直す用途向けです。GPU は不要です。しきい値を下げるとより多くの場面を拾います。

In [ ]:
from google.colab import files
from pathlib import Path
import shutil
import tempfile

if not shutil.which('ffmpeg'):
    raise RuntimeError('ffmpeg がありません。Colab のランタイムで実行してください。')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('動画を1つだけ選んでください。')
name, data = next(iter(uploaded.items()))
suffix = Path(name).suffix.lower()
if suffix not in {'.mp4', '.mov', '.mkv', '.webm'}:
    raise ValueError('MP4 / MOV / MKV / WEBM を選んでください。')
if len(data) > 200 * 1024**2:
    raise ValueError('200 MiB 以下の動画を選んでください。')
work_dir = Path(tempfile.mkdtemp(prefix='scene_sheet_', dir='/content'))
video_path = work_dir / ('input' + suffix)
video_path.write_bytes(data)
print('入力:', Path(name).name)


## シーンを抽出

しきい値は 0〜1、初期値 0.35。最大36枚までに制限します。フェードやカメラの揺れでは抽出数が増減します。

In [ ]:
import subprocess

SCENE_THRESHOLD = 0.35
MAX_SCENES = 36
THUMB_WIDTH = 320
if not 0 < SCENE_THRESHOLD < 1 or not 1 <= MAX_SCENES <= 100:
    raise ValueError('SCENE_THRESHOLD と MAX_SCENES の設定を確認してください。')

frame_dir = work_dir / 'frames'
frame_dir.mkdir()
base = ['ffmpeg', '-hide_banner', '-loglevel', 'error', '-i', str(video_path)]
subprocess.run(base + ['-frames:v', '1', '-vf', f'scale={THUMB_WIDTH}:-1',
                       '-y', str(frame_dir / 'frame_0000.png')], check=True)
if MAX_SCENES > 1:
    subprocess.run(base + ['-vf', rf"select=gt(scene\,{SCENE_THRESHOLD}),scale={THUMB_WIDTH}:-1",
                           '-vsync', 'vfr', '-frames:v', str(MAX_SCENES - 1),
                           '-y', str(frame_dir / 'frame_%04d.png')], check=True)
frames = sorted(frame_dir.glob('*.png'))
if not frames:
    raise RuntimeError('フレームを取り出せませんでした。動画を確認してください。')
print('抽出枚数:', len(frames))


## 一覧画像を作成してダウンロード

In [ ]:
from PIL import Image, ImageDraw
import math
import zipfile

PADDING = 12
LABEL_HEIGHT = 26
with Image.open(frames[0]) as sample:
    width, height = sample.size
columns = min(4, len(frames))
rows = math.ceil(len(frames) / columns)
sheet = Image.new('RGB', (columns * (width + PADDING) + PADDING,
                          rows * (height + LABEL_HEIGHT + PADDING) + PADDING), '#202020')
draw = ImageDraw.Draw(sheet)
for index, path in enumerate(frames):
    x = PADDING + index % columns * (width + PADDING)
    y = PADDING + index // columns * (height + LABEL_HEIGHT + PADDING)
    with Image.open(path) as im:
        sheet.paste(im.convert('RGB'), (x, y))
    draw.text((x, y + height + 3), f'{index + 1:02d}', fill='white')
sheet_path = work_dir / 'contact_sheet.jpg'
sheet.save(sheet_path, quality=90)
display(sheet)
zip_path = work_dir / 'scene_frames.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(sheet_path, arcname=sheet_path.name)
    for path in frames:
        archive.write(path, arcname=f'frames/{path.name}')
files.download(str(zip_path))
